# Data Cleaning and Visualization: 2008 Olympics Medalists

## Project Objective
The goal of this notebook is to clean the `olympics_08_medalists.csv` dataset by applying **Tidy Data Principles**. According to the [Tidy Data Paper](https://vita.had.co.nz/papers/tidy-data.pdf), a dataset is tidy when:
1. Each variable is in its own column.
2. Each observation forms its own row.
3. Each type of observational unit forms its own table.

We will use functions referenced in the [Pandas Cheat Sheet](https://pandas.pydata.org/Pandas_Cheat_Sheet.pdf), such as `pd.melt()` and `pivot_table()`, to reshape the data from a wide, untidy format into a long, tidy format for exploratory data analysis.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Load the dataset
df = pd.read_csv('olympics_08_medalists.csv')

# 2. Data Cleaning & Tidy Process
df_tidy = (
    # Melt: Collapse all sport/gender columns into one 'Gender_Sport' column
    df.melt(id_vars=['medalist_name'], var_name='Gender_Sport', value_name='Medal')
    # Drop rows where the athlete didn't win a medal
    .dropna(subset=['Medal'])
    .assign(
        # split(): Separate 'male_archery' into 'male'
        Gender = lambda x: x['Gender_Sport'].str.split('_').str[0].str.capitalize(),
        # replace(): Remove underscores and format the sport name
        Sport = lambda x: x['Gender_Sport'].str.split('_', n=1).str[1].str.replace('_', ' ').str.title()
    )
    # Rename for clarity
    .rename(columns={'medalist_name': 'Medalist'})
    # Reorder columns: Each variable is now in its own column
    [['Medalist', 'Gender', 'Sport', 'Medal']] 
)

print("Tidy DataFrame Preview:")
display(df_tidy.head()) # Use display() in Jupyter for pretty tables

## Exploratory Data Analysis: Pivot Table
Now that the data is tidy, we can easily aggregate it. We will use a pivot table to count the number of Gold, Silver, and Bronze medals awarded per Sport.

In [ ]:
# Create a pivot table to aggregate counts
pivot_medals = pd.pivot_table(
    df_tidy, 
    index='Sport', 
    columns='Medal', 
    values='Medalist', 
    aggfunc='count', 
    fill_value=0
)

# Sort by total medals
pivot_medals['Total'] = pivot_medals.sum(axis=1)
pivot_medals_sorted = pivot_medals.sort_values(by='Total', ascending=False)

print("Medal Counts Pivot Table (Top 10):")
display(pivot_medals_sorted[['Gold', 'Silver', 'Bronze']].head(10))

## Visualizations
The tidy dataset allows us to seamlessly pass our variables to `seaborn` for visualization. 
1. **Top 10 Sports by Total Medals**: Shows which events had the most medalists.
2. **Medal Distribution by Gender**: Compares the count of male vs. female medalists.

In [ ]:
# Visualization 1: Top 10 Sports by Medal Volume
plt.figure(figsize=(10, 6))
top_sports = df_tidy['Sport'].value_counts().head(10)
# Note: Added 'hue' and 'legend=False' to comply with seaborn v0.14.0 standards
sns.barplot(x=top_sports.values, y=top_sports.index, hue=top_sports.index, palette='magma', legend=False)
plt.title('Top 10 Sports by Total Medals Awarded (2008)')
plt.xlabel('Number of Medalists')
plt.ylabel('Sport')
plt.show()

# Visualization 2: Gender Distribution
plt.figure(figsize=(8, 5))
sns.countplot(data=df_tidy, x='Gender', hue='Gender', palette='Set2', legend=False)
plt.title('Total Medals Awarded by Gender')
plt.ylabel('Count of Athletes')
plt.show()